**Week 6 — Bias-Variance Trade-Off (พหุนามหลายดีกรี, Test MSE, Bias²/Variance, KNN)**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:**
  - `pb06_season.csv`: ข้อมูลฤดูกาล ($x$ = ช่วงเวลาของปี $0–1$, $y$ = ยอดขายรายสัปดาห์, $f(x) = 5 + 2\sin(2\pi x)$)
  - `pb06_pilot.csv`: ข้อมูลสาขานำร่อง ($x$ = ระดับส่วนลด $0–1$, $y$ = ยอดขายที่เพิ่มขึ้น)
  - `pb06_bv.csv`: ข้อมูลจำลอง 200 ชุดฝึก สำหรับการประมาณค่า Bias² และ Variance ที่ $x_0 = 0.75$
  - `pb06_seg.csv`: ข้อมูลจำลองการจำแนกลูกค้า ($x_1, x_2$, $y \in \{0, 1\}$)
  - `biz_customers.csv`: ข้อมูลลูกค้าสำหรับเปรียบเทียบพหุนามทำนายยอดซื้อจากอายุ
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์ ลำดับแถวคงเดิมตามไฟล์
- **การแบ่งชุดข้อมูล:** แบ่งตามตำแหน่งแถวที่ระบุในโจทย์ ไม่สุ่มข้อมูล
- **การสร้างโมเดลพหุนามดีกรี $d$:** ใช้ `make_pipeline(PolynomialFeatures(d), LinearRegression())` หรือ `np.polyfit(x, y, d)` โดย fit บนชุด train เท่านั้น
- **การวัดผล:** $\text{Test MSE} = \frac{1}{n_{\text{test}}} \sum (y_{\text{test}} - \hat{y}_{\text{test}})^2$

### รายละเอียดไฟล์ข้อมูล

**1. `pb06_season.csv`** (120 แถว)
- `x`: ช่วงเวลาของปี (ปรับสเกล 0–1)
- `y`: ยอดขายรายสัปดาห์ของสินค้าตามฤดูกาล (ร้อยชิ้น) สร้างจาก $y = 5 + 2\sin(2\pi x) + \epsilon$
- `set`: ระบุชุดข้อมูล (`train` แถวที่ 0–69 จำนวน 70 แถว, `test` แถวที่ 70–119 จำนวน 50 แถว)

**2. `pb06_pilot.csv`** (85 แถว)
- `x`: ระดับส่วนลดของโปรโมชัน (ปรับสเกล 0–1)
- `y`: ยอดขายที่เพิ่มขึ้นของสาขานำร่อง (หน่วยพันบาท)
- `set`: `train` (แถวที่ 0–24 จำนวน 25 แถว), `test` (แถวที่ 25–84 จำนวน 60 แถว)

**3. `pb06_bv.csv`** (200 แถว บันทึกผลจากชุดข้อมูลสุ่ม 200 ชุด)
- `sim`: หมายเลขรอบจำลอง (0–199)
- `pred_deg2`: ค่าทำนายที่ $x_0 = 0.75$ จากพหุนามดีกรี 2
- `pred_deg4`: ค่าทำนายที่ $x_0 = 0.75$ จากพหุนามดีกรี 4
- `pred_deg7`: ค่าทำนายที่ $x_0 = 0.75$ จากพหุนามดีกรี 7

**4. `pb06_seg.csv`** (300 แถว)
- `x1`: ความถี่ในการเข้าเว็บ/แอป (ปรับสเกลแล้ว)
- `x2`: รายได้ (ปรับสเกลแล้ว)
- `y`: การต่ออายุสมาชิก (1 = ต่ออายุ, 0 = ไม่ต่ออายุ)
- `set`: `train` (แถวที่ 0–199 จำนวน 200 แถว), `test` (แถวที่ 200–299 จำนวน 100 แถว)

**5. `biz_customers.csv`** (1203 แถว)
- `age`: อายุของลูกค้า (ปี)
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)

**โจทย์ข้อ 6.1:** กรณีศึกษาทีมวางแผนสินค้าของเครือร้านสะดวกซื้อต้องการรู้ว่าโมเดลที่ซับซ้อนขึ้นจะทำนายยอดขายตามฤดูกาลในข้อมูลใหม่ได้ดีเพียงใด ทีมใช้ไฟล์ `pb06_season.csv` (120 แถว คอลัมน์ `x` = ช่วงเวลาของปี ปรับสเกล 0–1, `y` = ยอดขายรายสัปดาห์ หน่วยร้อยชิ้น, `set` = `train` หรือ `test`) โดยแถวที่ 0–69 เป็นชุด train (`set = train`) และแถวที่ 70–119 เป็นชุด test (`set = test`) ให้ fit โมเดลพหุนามดีกรี 3 (ตัวทำนาย $x, x^2, x^3$ พร้อม intercept) ด้วยชุด train เท่านั้น แล้วคำนวณ MSE ของค่าทำนายบนชุด test (Test MSE) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

# 1. อ่านข้อมูล pb06_season.csv
df = pd.read_csv('pb06_season.csv')
train = df.iloc[:70]
test = df.iloc[70:]
print(f"ขนาดชุด train: {train.shape}, ขนาดชุด test: {test.shape}")

# 2. Fit โมเดลพหุนามดีกรี 3 ด้วยชุด train
poly3 = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
poly3.fit(train[['x']], train['y'])

# 3. คำนวณ Test MSE บนชุด test
pred_test = poly3.predict(test[['x']])
test_mse = mean_squared_error(test['y'], pred_test)
print(f"Test MSE (พหุนามดีกรี 3): {test_mse:.4f}")
print(f"คำตอบข้อ 6.1: {test_mse:.4f}")


ขนาดชุด train: (70, 3), ขนาดชุด test: (50, 3)
Test MSE (พหุนามดีกรี 3): 0.1579
คำตอบข้อ 6.1: 0.1579


**โจทย์ข้อ 6.2:** ฝึกถ้าเรารู้ฟังก์ชันจริง $f(x) = 5 + 2\sin(2\pi x)$ (มุมเป็นเรเดียน) ล่วงหน้าและใช้ $f(x)$ เป็นค่าทำนายของ $y$ โดยตรง (ไม่ต้อง fit อะไรเลย) MSE ของค่าทำนายนี้บนชุด test ของ `pb06_season.csv` (`set = test`) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# 1. อ่านชุด test ของ pb06_season.csv
df = pd.read_csv('pb06_season.csv')
test = df.iloc[70:]

# 2. คำนวณค่าจริง f(x) = 5 + 2*sin(2*pi*x)
f_true = 5.0 + 2.0 * np.sin(2 * np.pi * test['x'])

# 3. คำนวณ MSE ของ f(x) เทียบกับ y_test (Irreducible Error / Var(epsilon))
mse_true = mean_squared_error(test['y'], f_true)
print(f"MSE ของฟังก์ชันจริง f(x): {mse_true:.4f}")
print(f"คำตอบข้อ 6.2: {mse_true:.4f}")


MSE ของฟังก์ชันจริง f(x): 0.1209
คำตอบข้อ 6.2: 0.1209


**โจทย์ข้อ 6.3:** กรณีศึกษานักวิเคราะห์ของห้างสรรพสินค้าอยากแยกว่า error ของโมเดลพหุนามดีกรี 2 ที่ fit ด้วยชุด train ของ `pb06_season.csv` (`set = train`) บนชุด test (`set = test`) มีส่วนที่ "ลดได้" (reducible) เท่าใดเมื่อเทียบกับการใช้ฟังก์ชันจริง $f(x) = 5 + 2\sin(2\pi x)$ ทำนาย ให้หาค่า **(Test MSE ของโมเดลดีกรี 2) − (Test MSE ของการใช้ $f(x)$ ทำนาย)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

# 1. เตรียมชุด train และ test
df = pd.read_csv('pb06_season.csv')
train = df.iloc[:70]
test = df.iloc[70:]

# 2. Fit โมเดลพหุนามดีกรี 2
poly2 = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly2.fit(train[['x']], train['y'])
pred_deg2 = poly2.predict(test[['x']])
mse_deg2 = mean_squared_error(test['y'], pred_deg2)

# 3. MSE ของการใช้ f(x) ทำนาย
f_true = 5.0 + 2.0 * np.sin(2 * np.pi * test['x'])
mse_true = mean_squared_error(test['y'], f_true)

# 4. หา Reducible Error = Test MSE(deg 2) - Test MSE(f(x))
reducible_err = mse_deg2 - mse_true
print(f"Test MSE ดีกรี 2: {mse_deg2:.4f}")
print(f"Test MSE ของ f(x): {mse_true:.4f}")
print(f"Reducible Error: {reducible_err:.4f}")
print(f"คำตอบข้อ 6.3: {reducible_err:.4f}")


Test MSE ดีกรี 2: 0.9694
Test MSE ของ f(x): 0.1209
Reducible Error: 0.8485
คำตอบข้อ 6.3: 0.8485


**โจทย์ข้อ 6.4:** ฝึก fit โมเดลพหุนามดีกรี 4 ด้วยชุด train ของ `pb06_season.csv` (ตัวทำนายคือ $x, \dots, x^4$ และมี intercept) สัมประสิทธิ์ของพจน์ $x^4$ (พจน์ที่มีกำลังสูงสุด) = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# 1. ใช้ชุด train แถวที่ 0-69
df = pd.read_csv('pb06_season.csv')
train = df.iloc[:70]

# 2. Fit โมเดลพหุนามดีกรี 4 ด้วย make_pipeline
poly4 = make_pipeline(PolynomialFeatures(degree=4), LinearRegression())
poly4.fit(train[['x']], train['y'])

# 3. สัมประสิทธิ์ของ x^4 (ลำดับพจน์: 1, x, x^2, x^3, x^4)
coefs = poly4.named_steps['linearregression'].coef_
c4 = coefs[4]
print(f"สัมประสิทธิ์ทุกพจน์: {coefs}")
print(f"สัมประสิทธิ์พจน์ x^4: {c4:.2f}")
print(f"คำตอบข้อ 6.4: {c4:.2f}")


สัมประสิทธิ์ทุกพจน์: [  0.          23.67531741 -62.07108057  28.42667362  11.01313423]
สัมประสิทธิ์พจน์ x^4: 11.01
คำตอบข้อ 6.4: 11.01


**โจทย์ข้อ 6.5:** กรณีศึกษาฝ่ายการตลาดทดลองโปรโมชันส่วนลดกับสาขานำร่อง มีผลวัดระดับส่วนลด $x$ (ปรับสเกล 0–1) กับยอดขายที่เพิ่มขึ้น $y$ (หน่วยพันบาท) ของ 85 สาขา ในไฟล์ `pb06_pilot.csv` (แถวละ 1 สาขา) ข้อมูลที่ติดป้ายแล้วมีน้อย จึงใช้เพียง 25 แถวแรก (แถวที่ 0–24, `set = train`) เป็นชุด train และอีก 60 แถวที่เหลือ (`set = test`) เป็นชุด test ให้ fit โมเดลพหุนามดีกรี 7 ด้วยชุด train แล้วคำนวณ Test MSE บนชุด test ทั้งหมด = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

# 1. อ่านข้อมูล pb06_pilot.csv และแบ่ง train (25 แถว) / test (60 แถว)
df = pd.read_csv('pb06_pilot.csv')
train = df.iloc[:25]
test = df.iloc[25:]
print(f"ขนาด Train: {train.shape}, ขนาด Test: {test.shape}")

# 2. Fit โมเดลพหุนามดีกรี 7 ด้วยชุด train
poly7 = make_pipeline(PolynomialFeatures(degree=7), LinearRegression())
poly7.fit(train[['x']], train['y'])

# 3. คำนวณ Test MSE
pred_test = poly7.predict(test[['x']])
test_mse = mean_squared_error(test['y'], pred_test)
print(f"Test MSE (พหุนามดีกรี 7): {test_mse:.4f}")
print(f"คำตอบข้อ 6.5: {test_mse:.4f}")


ขนาด Train: (25, 3), ขนาด Test: (60, 3)
Test MSE (พหุนามดีกรี 7): 0.7240
คำตอบข้อ 6.5: 0.7240


**โจทย์ข้อ 6.6:** กรณีศึกษาทีมวิทยาศาสตร์ข้อมูลของแพลตฟอร์มอีคอมเมิร์ซจำลองการเก็บข้อมูลยอดขายตามฤดูกาล 200 ครั้ง (ไฟล์ `pb06_bv.csv`) แล้ว fit โมเดลพหุนามดีกรี 2 (คอลัมน์ `pred_deg2`) ทุกครั้ง เพื่อดูว่าโมเดลที่เรียบเกินไปนี้ทำนายที่ $x_0 = 0.75$ "เอนเอียง"จากค่าจริงมากเพียงใด ฟังก์ชันจริงคือ $f(x) = 5 + 2\sin(2\pi x)$ (มุมเป็นเรเดียน) ให้หา $\text{Bias}^2 = (\text{ค่าเฉลี่ยของค่าทำนาย 200 ครั้ง} - f(x_0))^2$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลผลจำลอง 200 ครั้ง
bv_df = pd.read_csv('pb06_bv.csv')

# 2. คำนวณค่าจริง f(x0) ที่ x0 = 0.75
x0 = 0.75
f_x0 = 5.0 + 2.0 * np.sin(2 * np.pi * x0)
print(f"ค่าจริง f(0.75): {f_x0}")

# 3. ค่าเฉลี่ยของค่าทำนาย 200 ครั้งของพหุนามดีกรี 2
mean_pred_deg2 = bv_df['pred_deg2'].mean()
print(f"ค่าเฉลี่ยของ pred_deg2: {mean_pred_deg2:.4f}")

# 4. คำนวณ Bias^2
bias2 = (mean_pred_deg2 - f_x0)**2
print(f"Bias^2: {bias2:.4f}")
print(f"คำตอบข้อ 6.6: {bias2:.4f}")


ค่าจริง f(0.75): 3.0
ค่าเฉลี่ยของ pred_deg2: 3.9192
Bias^2: 0.8449
คำตอบข้อ 6.6: 0.8449


**โจทย์ข้อ 6.7:** ฝึกจาก `pb06_bv.csv` ความแปรปรวน (Variance) ของค่าทำนายของพหุนามดีกรี 7 (คอลัมน์ `pred_deg7`) ที่ $x_0 = 0.75$ คือความแปรปรวนของค่าทำนายทั้ง 200 ครั้ง โดยหารด้วย 200 (ค่าเริ่มต้นของ `np.var`) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ pb06_bv.csv
bv_df = pd.read_csv('pb06_bv.csv')

# 2. คำนวณ Variance ของ pred_deg7 (ddof=0)
var_deg7 = np.var(bv_df['pred_deg7'], ddof=0)
print(f"Variance ของค่าทำนายดีกรี 7: {var_deg7:.4f}")
print(f"คำตอบข้อ 6.7: {var_deg7:.4f}")


Variance ของค่าทำนายดีกรี 7: 0.5411
คำตอบข้อ 6.7: 0.5411


**โจทย์ข้อ 6.8:** กรณีศึกษาทีมวางแผนสินค้าของศูนย์กระจายสินค้าต้องการประเมินค่าคาดหวังของ Test MSE ณ จุด $x_0 = 0.75$ ของโมเดลพหุนามดีกรี 4 (คอลัมน์ `pred_deg4` ใน `pb06_bv.csv`) โดยแยกเป็น $\text{Bias}^2 + \text{Variance} + \text{Var}(\epsilon)$ ให้ใช้ฟังก์ชันจริง $f(x) = 5 + 2\sin(2\pi x)$ (มุมเป็นเรเดียน), Variance แบบหารด้วย 200 (ค่าเริ่มต้นของ `np.var`) และ $\text{Var}(\epsilon) = 0.25$ ค่าคาดหวังของ Test MSE ที่ $x_0$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ pb06_bv.csv
bv_df = pd.read_csv('pb06_bv.csv')

# 2. คำนวณ Bias^2 ของดีกรี 4 ที่ x0 = 0.75
x0 = 0.75
f_x0 = 5.0 + 2.0 * np.sin(2 * np.pi * x0)
mean_pred4 = bv_df['pred_deg4'].mean()
bias2_deg4 = (mean_pred4 - f_x0)**2

# 3. คำนวณ Variance ของดีกรี 4 (ddof=0)
var_deg4 = np.var(bv_df['pred_deg4'], ddof=0)

# 4. รวม Bias^2 + Variance + Var(epsilon)
var_eps = 0.25
expected_test_mse = bias2_deg4 + var_deg4 + var_eps
print(f"Bias^2: {bias2_deg4:.6f}")
print(f"Variance: {var_deg4:.6f}")
print(f"Var(epsilon): {var_eps:.4f}")
print(f"Expected Test MSE: {expected_test_mse:.4f}")
print(f"คำตอบข้อ 6.8: {expected_test_mse:.4f}")


Bias^2: 0.000030
Variance: 0.113487
Var(epsilon): 0.2500
Expected Test MSE: 0.3635
คำตอบข้อ 6.8: 0.3635


**โจทย์ข้อ 6.9:** ฝึกใช้ `pb06_seg.csv` (แถวที่ 0–199 คือ `set = train` และแถวที่ 200–299 คือ `set = test`) fit `KNeighborsClassifier(n_neighbors=11)` ด้วยชุด train โดยใช้ $X$ = `x1` และ `x2` (ตามลำดับนี้) และ $y$ = `y` (0/1) แล้วหา Test error rate (สัดส่วนแถวในชุด test ที่ทำนายผิด = $1 - \text{accuracy}$) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. อ่านข้อมูลจำแนกกลุ่ม pb06_seg.csv
df = pd.read_csv('pb06_seg.csv')
train = df.iloc[:200]
test = df.iloc[200:]
print(f"ขนาด Train: {train.shape}, ขนาด Test: {test.shape}")

# 2. Fit KNeighborsClassifier (k=11)
knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(train[['x1', 'x2']], train['y'])

# 3. ทำนายและหา Test error rate = 1 - accuracy
pred_test = knn.predict(test[['x1', 'x2']])
acc = accuracy_score(test['y'], pred_test)
error_rate = 1.0 - acc
print(f"Accuracy บนชุด test: {acc:.4f}")
print(f"Test error rate: {error_rate:.4f}")
print(f"คำตอบข้อ 6.9: {error_rate:.4f}")


ขนาด Train: (200, 4), ขนาด Test: (100, 4)
Accuracy บนชุด test: 0.7600
Test error rate: 0.2400
คำตอบข้อ 6.9: 0.2400


**โจทย์ข้อ 6.10:** กรณีศึกษาฝ่าย CRM ของธนาคารที่มีบัตรเครดิตร่วมกับร้านค้าสงสัยว่ายอดซื้อต่อเดือน (`spend_k`) ของลูกค้าไม่ได้เปลี่ยนตามอายุ (`age`) เป็นเส้นตรง จึงเทียบโมเดลเชิงเส้น (พหุนามดีกรี 1) กับโมเดลพหุนามดีกรี 3 (ตัวทำนายคืออายุ ยกกำลัง 1 ถึง 3 พร้อม intercept ใช้ค่า `age` ตามไฟล์ `biz_customers.csv` ไม่ต้องปรับสเกล) โดยใช้ 960 แถวแรกของไฟล์ (แถวที่ 0–959) เป็นชุด train และ 243 แถวที่เหลือ (แถวที่ 960–1202) เป็นชุด test ให้หาว่า **Test MSE ของโมเดลดีกรี 3 ลดลงกี่เปอร์เซ็นต์เมื่อเทียบกับ Test MSE ของโมเดลเชิงเส้น** คือ $(\text{Test MSE ดีกรี 1} - \text{Test MSE ดีกรี 3}) \div (\text{Test MSE ดีกรี 1}) \times 100$ = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

# 1. อ่านข้อมูลลูกค้าและแบ่ง train (960 คน) / test (243 คน)
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

# 2. Fit โมเดลพหุนามดีกรี 1 (Linear Regression)
deg1 = make_pipeline(PolynomialFeatures(degree=1), LinearRegression())
deg1.fit(train[['age']], train['spend_k'])
pred_deg1 = deg1.predict(test[['age']])
mse_deg1 = mean_squared_error(test['spend_k'], pred_deg1)

# 3. Fit โมเดลพหุนามดีกรี 3
deg3 = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
deg3.fit(train[['age']], train['spend_k'])
pred_deg3 = deg3.predict(test[['age']])
mse_deg3 = mean_squared_error(test['spend_k'], pred_deg3)

# 4. คำนวณเปอร์เซ็นต์ที่ Test MSE ลดลง
pct_reduction = (mse_deg1 - mse_deg3) / mse_deg1 * 100.0
print(f"Test MSE (ดีกรี 1): {mse_deg1:.4f}")
print(f"Test MSE (ดีกรี 3): {mse_deg3:.4f}")
print(f"เปอร์เซ็นต์การลดลงของ Test MSE: {pct_reduction:.2f}%")
print(f"คำตอบข้อ 6.10: {pct_reduction:.2f}")


Test MSE (ดีกรี 1): 2.3498
Test MSE (ดีกรี 3): 2.2055
เปอร์เซ็นต์การลดลงของ Test MSE: 6.14%
คำตอบข้อ 6.10: 6.14


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 6

---

### 1. การแยกส่วนความคลาดเคลื่อน (Bias-Variance Decomposition)

สำหรับจุดทดสอบ $x_0$ ภายใต้แบบจำลอง $Y = f(x_0) + \epsilon$ โดย $\mathbb{E}[\epsilon] = 0$ และ $\text{Var}(\epsilon) = \sigma^2$ ค่าคาดหวังของ Test MSE จากการสุ่มชุดฝึกหลายๆ ครั้ง สามารถแยกออกเป็น 3 ส่วนหลัก:
$$\mathbb{E}\left[(Y - \hat{f}(x_0))^2\right] = \text{Bias}(\hat{f}(x_0))^2 + \text{Var}(\hat{f}(x_0)) + \text{Var}(\epsilon)$$

1. **$\text{Bias}(\hat{f}(x_0))^2 = (\mathbb{E}[\hat{f}(x_0)] - f(x_0))^2$ (ความเอนเอียงกำลังสอง):**
   - เกิดจากการที่โมเดลมีสมมติฐานที่เรียบง่ายเกินไป ไม่สามารถจับรูปแบบความสัมพันธ์ที่แท้จริงได้ (Underfitting)
   - เช่น การใช้พหุนามดีกรี 1 หรือ 2 ทำนายฟังก์ชันที่เป็นคลื่น $\sin(2\pi x)$

2. **$\text{Var}(\hat{f}(x_0)) = \mathbb{E}[(\hat{f}(x_0) - \mathbb{E}[\hat{f}(x_0)])^2]$ (ความแปรปรวน):**
   - วัดว่าค่าทำนายเปลี่ยนไปมากน้อยเพียงใดเมื่อเปลี่ยนชุดข้อมูลฝึก
   - โมเดลที่ซับซ้อนมาก (เช่น พหุนามดีกรี 7) มีความยืดหยุ่นสูง จะมี Variance สูง ไวต่อจุดข้อมูลในชุดฝึกแต่ละชุดมาก (Overfitting)

3. **$\text{Var}(\epsilon) = \sigma^2$ (Irreducible Error):**
   - สัญญาณรบกวนหรือ noise ที่ติดมากับกระบวนการกำเนิดข้อมูลตามธรรมชาติ แม้เราจะรู้ฟังก์ชันจริง $f(x)$ ก็ไม่สามารถลดความคลาดเคลื่อนส่วนนี้ให้ต่ำกว่า $\sigma^2$ ได้

---

### 2. Reducible Error vs Irreducible Error

$$\text{Test MSE} = \underbrace{\text{Bias}^2 + \text{Variance}}_{\text{Reducible Error (ลดทอนได้ด้วยการเลือกโมเดลที่เหมาะสม)}} + \underbrace{\text{Var}(\epsilon)}_{\text{Irreducible Error}}$$

- หากใช้ฟังก์ชันจริง $f(x)$ ในการทำนาย Reducible Error จะเป็นศูนย์ เหลือเพียง Irreducible Error $\text{Var}(\epsilon)$
- ผลต่าง $(\text{Test MSE ของโมเดล}) - (\text{Test MSE ของ } f(x))$ จึงแสดงถึงความคลาดเคลื่อนส่วนที่สามารถลดลงได้ หากปรับปรุงโมเดลให้มีความซับซ้อนที่เหมาะสม

---

### 3. Bias-Variance Trade-Off และความซับซ้อนของโมเดล (Model Complexity)

| ดีกรีพหุนาม ($d$) | ความซับซ้อน | Bias | Variance | ปรากฏการณ์ | Test MSE |
| :---: | :---: | :---: | :---: | :---: | :---: |
| ต่ำ ($d=1, 2$) | ต่ำ (Inflexible) | สูง (High Bias) | ต่ำ (Low Var) | **Underfitting** | สูง (Bias นำ) |
| เหมาะสม ($d=3, 4$) | ปานกลาง | ต่ำ | ต่ำ | **Sweet Spot** | **ต่ำที่สุด** |
| สูง ($d=7$) | สูง (Over-flexible) | ต่ำมาก | สูงมาก (High Var) | **Overfitting** | สูง (Variance นำ) |

---

### 4. การวัดผลในงานจำแนกประเภท (Classification Error Rate)

ในงานจำแนกประเภท ประเมินประสิทธิภาพบนชุดทดสอบด้วย **Test Error Rate**:
$$\text{Test Error Rate} = \frac{1}{n_{\text{test}}} \sum_{i=1}^{n_{\text{test}}} \mathbb{I}(y_i \ne \hat{y}_i) = 1 - \text{Accuracy}$$
อัลกอริทึม KNN Classifier จะถูกควบคุมระดับความซับซ้อนด้วยค่า $K$ โดย $K$ มากจะให้ขอบเขตการตัดสินใจที่เรียบ (Low Variance, High Bias) ขณะที่ $K$ น้อยจะให้ขอบเขตที่หยักซับซ้อน (High Variance, Low Bias)